# 3D NIfTI Reptile Few-Shot Pipeline

This notebook uses `config.py`, `resize.py`, `split.py`, `dataloader_ben2.py`, and `model.py`.

## Section 0.1: Install Dependencies


In [ ]:
import sys

INSTALL_DEPS = False

if INSTALL_DEPS:
    !{sys.executable} -m pip install nibabel scikit-image tqdm matplotlib numpy torch
else:
    print("Skipping dependency install. Set INSTALL_DEPS = True if imports fail.")


## Section 1: Read Config

In [ ]:
import config

print("Data directory:", config.DATA_DIR)
print("Resized data directory:", config.RESIZED_DATA_DIR)
print("Resize dimensions:", config.RESIZE_DIMS)
print("Batch size:", config.BATCH_SIZE)
print("UNet features:", config.UNET_FEATURES)
print("Reptile outer steps:", config.REPTILE_OUTER_STEPS)
print("Reptile inner steps:", config.REPTILE_INNER_STEPS)
print("Reptile inner LR:", config.REPTILE_INNER_LR)
print("Reptile outer LR:", config.REPTILE_OUTER_LR)

## Section 2: Initialise Functions

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import torch

from model import UNet3D, bce_dice_loss, dice_score, prepare_episode, validate, test, get_device
from dataloader_ben2 import build_3d_dataloader, build_episode_loader


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = get_device()
print("Python:", sys.executable)
print("Device:", device)

## Section 3: Resize Dataset And Split

In [ ]:
RUN_RESIZE = True

if RUN_RESIZE:
    !{sys.executable} resize.py --input-dir {config.DATA_DIR} --output-dir {config.RESIZED_DATA_DIR} --x {config.RESIZE_DIMS[0]} --y {config.RESIZE_DIMS[1]} --depth {config.RESIZE_DIMS[2]}
else:
    print("Skipping resize. Set RUN_RESIZE = True to regenerate data-resize/.")

In [ ]:
RUN_SPLIT = True

if RUN_SPLIT:
    !{sys.executable} split.py --data-dir {config.RESIZED_DATA_DIR} --output-dir . --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15 --seed 42
else:
    print("Skipping split generation.")

## Section 4: Reptile

In [ ]:
model = UNet3D().to(device)
print(model.__class__.__name__)

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    !{sys.executable} train_reptile.py --data-dir {config.RESIZED_DATA_DIR} --train-split train.txt --val-split val.txt --test-split test.txt --n-support 1 --n-query 1 --outer-steps {config.REPTILE_OUTER_STEPS} --inner-steps {config.REPTILE_INNER_STEPS} --inner-lr {config.REPTILE_INNER_LR} --outer-lr {config.REPTILE_OUTER_LR} --val-interval {config.REPTILE_VAL_INTERVAL} --batch-size {config.BATCH_SIZE} --eval-episodes 20 --model-path reptile_3d_unet.pth --history-path reptile_history.json --metrics-path reptile_metrics.json
else:
    print("Skipping training. Set RUN_TRAINING = True to call train_reptile.py.")

In [ ]:
import json

model_path = Path("reptile_3d_unet.pth")
history_path = Path("reptile_history.json")
metrics_path = Path("reptile_metrics.json")

history = {"support_loss": [], "query_loss": [], "val_dice": []}
if history_path.exists():
    history = json.loads(history_path.read_text())

model_loaded = model_path.exists()
if model_loaded:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
else:
    print("No saved Reptile model found yet.")

if metrics_path.exists():
    saved_metrics = json.loads(metrics_path.read_text())
    val_metrics = saved_metrics.get("validation", {})
    test_metrics = saved_metrics.get("test", {})
else:
    val_loader = build_3d_dataloader(config.RESIZED_DATA_DIR, "val.txt", config.BATCH_SIZE, shuffle=False)
    test_loader = build_3d_dataloader(config.RESIZED_DATA_DIR, "test.txt", config.BATCH_SIZE, shuffle=False)
    val_metrics = validate(model, val_loader, device=device)
    test_metrics = test(model, test_loader, device=device)

print("Validation:", val_metrics)
print("Test:", test_metrics)

## Section 5: Metrics

In [ ]:
import matplotlib.pyplot as plt

results = {
    "Reptile": {
        "validation": val_metrics,
        "test": test_metrics,
    }
}

print("=" * 78)
print(f"{'Method':<18} {'Val Dice':>10} {'Val Loss':>10} {'Test Dice':>10} {'Test Loss':>10}")
print("=" * 78)
print(
    f"{'Reptile':<18} "
    f"{val_metrics.get('dice', float('nan')):>10.3f} "
    f"{val_metrics.get('loss', float('nan')):>10.3f} "
    f"{test_metrics.get('dice', float('nan')):>10.3f} "
    f"{test_metrics.get('loss', float('nan')):>10.3f}"
)
print("=" * 78)


In [ ]:
if history["support_loss"]:
    fig, ax = plt.subplots(figsize=(14, 5))
    for key, colour in [("support_loss", "steelblue"), ("query_loss", "darkorange")]:
        values = np.array(history[key], dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            continue
        window = max(1, len(values) // 100)
        smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
        ax.plot(range(window - 1, len(values)), smoothed, label=key.replace("_", " "), color=colour)

    ax.set_xlabel("Outer step")
    ax.set_ylabel("Loss")
    ax.set_title("Reptile Meta-Training Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training history yet.")


In [ ]:
labels = ["Validation", "Test"]
means = [val_metrics.get("dice", np.nan), test_metrics.get("dice", np.nan)]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(labels, means, color=["steelblue", "darkorange"], alpha=0.85)
for bar, mean in zip(bars, means):
    if np.isfinite(mean):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{mean:.3f}", ha="center")
ax.set_ylabel("Dice Score")
ax.set_ylim(0, 1.0)
ax.set_title("Reptile Few-Shot Validation/Test Dice")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("saved_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import copy
import torch.optim as optim

RUN_VISUALISE = True


def visualise_reptile_prediction(base_model, n_support=1, n_query=3, inner_steps=config.REPTILE_INNER_STEPS, inner_lr=config.REPTILE_INNER_LR):
    if not model_loaded:
        print("Warning: no saved model was loaded. This prediction is from an untrained model.")

    episode = next(build_episode_loader(
        data_dir=config.RESIZED_DATA_DIR,
        split_file="test.txt",
        n_support=n_support,
        n_query=n_query,
        episodes=1,
    ))

    support_ids = episode["support"]["case_id"]
    query_ids = episode["query"]["case_id"]
    print("Support case(s):", support_ids)
    print("Query case(s):", query_ids)

    support_images, support_masks, query_images, query_masks = prepare_episode(episode, device)

    adapted_model = copy.deepcopy(base_model).to(device)
    adapted_model.train()
    optimizer = optim.SGD(adapted_model.parameters(), lr=inner_lr)

    for _ in range(inner_steps):
        optimizer.zero_grad()
        loss = bce_dice_loss(adapted_model(support_images), support_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), 1.0)
        optimizer.step()

    adapted_model.eval()
    with torch.no_grad():
        logits = adapted_model(query_images)
        probabilities = torch.sigmoid(logits).cpu()
        predictions = (probabilities > 0.5).float()

    query_images = query_images.cpu()
    query_masks = query_masks.cpu()
    n_rows = min(n_query, query_images.shape[0])

    fig, axes = plt.subplots(n_rows, 4, figsize=(13, 3.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row in range(n_rows):
        case_id = query_ids[row]
        gt_volume = query_masks[row, 0]
        pred_volume = predictions[row, 0]
        prob_volume = probabilities[row, 0]

        slice_scores = gt_volume.sum(dim=(1, 2))
        slice_idx = int(torch.argmax(slice_scores)) if torch.max(slice_scores).item() > 0 else gt_volume.shape[0] // 2

        intersection = (pred_volume * gt_volume).sum()
        union = pred_volume.sum() + gt_volume.sum()
        dice = ((2.0 * intersection + 1e-6) / (union + 1e-6)).item()
        pred_fraction = pred_volume.mean().item()

        image_slice = query_images[row, 0, slice_idx]
        mask_slice = gt_volume[slice_idx]
        prob_slice = prob_volume[slice_idx]
        pred_slice = pred_volume[slice_idx]

        titles = [
            f"Input\n{case_id} | slice {slice_idx}",
            f"Ground Truth\n{case_id}",
            f"Probability\nDice {dice:.3f}",
            f"Prediction > 0.5\nFG {pred_fraction:.1%}",
        ]
        cmaps = ["gray", "gray", "magma", "gray"]
        for ax, data, title, cmap in zip(axes[row], [image_slice, mask_slice, prob_slice, pred_slice], titles, cmaps):
            ax.imshow(data, cmap=cmap)
            ax.set_title(title, fontsize=9)
            ax.axis("off")

    fig.suptitle(f"Adapted using support case(s): {', '.join(support_ids)}", fontsize=11)
    plt.tight_layout()
    plt.savefig("qualitative_reptile_predictions.png", dpi=150, bbox_inches="tight")
    plt.show()


if RUN_VISUALISE:
    visualise_reptile_prediction(model, n_support=1, n_query=3)
else:
    print("Skipping qualitative prediction visualisation. Set RUN_VISUALISE = True to display query case IDs.")


In [ ]:
print("Final validation metrics:", val_metrics)
print("Final test metrics:", test_metrics)
